# NLP Models & Algorithms

In this notebook, we cover:
1. N-gram Language Modeling with Laplace (Add-1) Smoothing and Perplexity
2. Hidden Markov Models (HMM) & Viterbi Decoding for Part-of-Speech Tagging
3. Multinomial Naive Bayes Classifier for Intent Routing
4. Comprehensive Evaluation Metrics: Accuracy, Precision, Recall, Macro F1, Confusion Matrix


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from nlp_engine.data.generator import generate_support_tickets
from nlp_engine.probabilistic.ngram import NGramLanguageModel
from nlp_engine.probabilistic.hmm import HiddenMarkovModelTagger
from nlp_engine.probabilistic.classifier import TicketNaiveBayes
from nlp_engine.representation.tfidf import TFIDFModel
from nlp_engine.utils.metrics import calculate_classification_metrics

## 1. N-Gram Language Model with Smoothing & Perplexity


In [ ]:
sentences = [
    ["please", "refund", "my", "order"],
    ["can", "i", "cancel", "my", "subscription"],
    ["where", "is", "my", "package", "shipment"],
    ["please", "help", "with", "my", "account", "login"]
]
lm = NGramLanguageModel(n=2, alpha=1.0)
lm.fit(sentences)
test_query = ["please", "refund", "my", "order"]
print("Sentence Perplexity:", lm.calculate_sentence_perplexity(test_query))

## 2. HMM POS Tagger with Viterbi Algorithm


In [ ]:
tagged_data = [
    [("refund", "VERB"), ("the", "DET"), ("charge", "NOUN")],
    [("the", "DET"), ("charge", "NOUN"), ("was", "VERB"), ("disputed", "ADJ")],
    [("please", "ADV"), ("cancel", "VERB"), ("my", "PRON"), ("order", "NOUN")]
]
hmm = HiddenMarkovModelTagger()
hmm.fit(tagged_data)
print("Viterbi Decoded POS Tags:")
print(hmm.viterbi(["please", "refund", "the", "charge"]))

## 3. Naive Bayes Intent Routing & Evaluation


In [ ]:
df = generate_support_tickets(n_samples=250, random_state=42)
tfidf = TFIDFModel(max_features=500)
X = tfidf.fit_transform(df["text"].tolist())
y = df["category"].values

split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

nb = TicketNaiveBayes(alpha=1.0)
nb.fit(X_train, y_train)
preds = nb.predict(X_test)

metrics = calculate_classification_metrics(y_test, preds)
print(f"Accuracy  : {metrics['accuracy']*100:.2f}%")
print(f"Macro F1  : {metrics['macro_f1']:.4f}")